# 04 — Эксперименты с моделями

## Импорты и настройки окружения

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

import torch
from scipy.sparse import hstack
import scipy.sparse as sp

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import normalize
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from catboost import CatBoostClassifier
import lightgbm as lgb

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

os.environ['CUDA_VISIBLE_DEVICES'] = '3'
os.environ['LIGHTGBM_VERBOSE']     = '-1'

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


Device: cuda:0


## Загрузка данных

In [5]:
df_train = pd.read_csv('../CP2/train.csv')
df_val   = pd.read_csv('../CP2/val.csv')
df_test  = pd.read_csv('../CP2/test.csv')

X_train_text = df_train['text_clean']
y_train      = df_train['label']
X_val_text   = df_val['text_clean']
y_val        = df_val['label']
X_test_text  = df_test['text_clean']
y_test       = df_test['label']

FEAT_COLS = [c for c in df_train.columns if c.startswith('feat_')]
print(f'Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}')
print(f'Числовых фич: {len(FEAT_COLS)} -> {FEAT_COLS}')


Train: 37104, Val: 6548, Test: 14540
Числовых фич: 7 -> ['feat_char_len', 'feat_word_count', 'feat_upper_ratio', 'feat_exclamation_count', 'feat_question_count', 'feat_emoji_count', 'feat_avg_word_len']


## TF-IDF векторизация

In [6]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_val_tfidf   = tfidf.transform(X_val_text)
X_test_tfidf  = tfidf.transform(X_test_text)

print(f'TF-IDF матрица: {X_train_tfidf.shape}')


TF-IDF матрица: (37104, 50000)


## Функция evaluate и реестр моделей

In [ ]:
results        = []   
model_registry = {}  

def evaluate(name, model, X_tr, y_tr, X_v, y_v, hypothesis, params_note):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_v)

    f1_macro    = f1_score(y_v, preds, average='macro')
    f1_weighted = f1_score(y_v, preds, average='weighted')
    acc         = accuracy_score(y_v, preds)

    print(f'[{name}] Macro F1 = {f1_macro:.4f} | Weighted F1 = {f1_weighted:.4f} | Acc = {acc:.4f}')

    entry = {
        'Модель':           name,
        'Гипотеза':         hypothesis,
        'Параметры':        params_note,
        'Val Macro F1':     round(f1_macro, 4),
        'Val Weighted F1':  round(f1_weighted, 4),
        'Val Accuracy':     round(acc, 4),
    }
    results.append(entry)
    model_registry[name] = model

    df_temp = pd.DataFrame([entry])
    if not os.path.exists('experiments_log.csv'):
        df_temp.to_csv('experiments_log.csv', index=False)
    else:
        df_temp.to_csv('experiments_log.csv', mode='a', header=False, index=False)

    print('\nClassification Report:')
    print(classification_report(y_v, preds, target_names=['negative', 'neutral', 'positive']))
    print('-' * 80)

    return model, preds


## Эксперимент 1 — KNN


In [ ]:
X_train_norm = normalize(X_train_tfidf)
X_val_norm   = normalize(X_val_tfidf)

knn = KNeighborsClassifier(n_neighbors=7, metric='cosine', n_jobs=-1)
evaluate(
    'KNN (k=7, cosine)', knn,
    X_train_norm, y_train, X_val_norm, y_val,
    'Похожие тексты -> одинаковая тональность', 'k=7, cosine'
)

[KNN (k=7, cosine)] Macro F1 = 0.6621 | Weighted F1 = 0.6615 | Acc = 0.6590

Classification Report:
              precision    recall  f1-score   support

    negative       0.69      0.59      0.64      2224
     neutral       0.53      0.61      0.56      2173
    positive       0.79      0.78      0.79      2151

    accuracy                           0.66      6548
   macro avg       0.67      0.66      0.66      6548
weighted avg       0.67      0.66      0.66      6548

--------------------------------------------------------------------------------


(KNeighborsClassifier(metric='cosine', n_jobs=-1, n_neighbors=7),
 array([0, 2, 1, ..., 2, 1, 0], shape=(6548,)))

## Эксперимент 2 — Random Forest


In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1, class_weight='balanced')
evaluate(
    'Random Forest', rf,
    X_train_tfidf, y_train, X_val_tfidf, y_val,
    'Ансамбль деревьев лучше LogReg на тексте', 'n=200, balanced'
)

[Random Forest] Macro F1 = 0.7165 | Weighted F1 = 0.7161 | Acc = 0.7159

Classification Report:
              precision    recall  f1-score   support

    negative       0.75      0.67      0.71      2224
     neutral       0.60      0.64      0.62      2173
    positive       0.81      0.84      0.82      2151

    accuracy                           0.72      6548
   macro avg       0.72      0.72      0.72      6548
weighted avg       0.72      0.72      0.72      6548

--------------------------------------------------------------------------------


(RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                        random_state=42),
 array([0, 2, 0, ..., 2, 1, 0], shape=(6548,)))

## Эксперимент 3 — LightGBM



In [ ]:
lgbm = lgb.LGBMClassifier(
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=127,
    max_bin=63,
    min_data_in_leaf=20,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    device='gpu',
    gpu_platform_id=0,
    gpu_device_id=0,
    random_state=SEED,
    verbose=-1,
    n_jobs=1         
)
evaluate(
    'LightGBM', lgbm,
    X_train_tfidf, y_train, X_val_tfidf, y_val,
    'Бустинг лучше случайного леса', 'n=800, lr=0.05, leaves=127, GPU'
)

[LightGBM] Macro F1 = 0.7207 | Weighted F1 = 0.7202 | Acc = 0.7193

Classification Report:
              precision    recall  f1-score   support

    negative       0.71      0.71      0.71      2224
     neutral       0.60      0.62      0.61      2173
    positive       0.85      0.83      0.84      2151

    accuracy                           0.72      6548
   macro avg       0.72      0.72      0.72      6548
weighted avg       0.72      0.72      0.72      6548

--------------------------------------------------------------------------------


(LGBMClassifier(bagging_fraction=0.8, bagging_freq=5, device='gpu',
                feature_fraction=0.8, gpu_device_id=0, gpu_platform_id=0,
                learning_rate=0.05, max_bin=63, min_data_in_leaf=20,
                n_estimators=800, n_jobs=1, num_leaves=127, random_state=42,
                verbose=-1),
 array([0, 2, 0, ..., 0, 1, 0], shape=(6548,)))

## Эксперимент 4 — CatBoost



In [ ]:
cb = CatBoostClassifier(
    iterations=800,
    learning_rate=0.05,
    depth=8,
    task_type='GPU',
    devices='0',
    random_seed=SEED,
    verbose=0
)
evaluate(
    'CatBoost', cb,
    X_train_tfidf, y_train, X_val_tfidf, y_val,
    'CatBoost конкурентен с LightGBM', 'iter=800, lr=0.05, depth=8, GPU'
)

[CatBoost] Macro F1 = 0.7294 | Weighted F1 = 0.7290 | Acc = 0.7260

Classification Report:
              precision    recall  f1-score   support

    negative       0.76      0.69      0.72      2224
     neutral       0.60      0.68      0.64      2173
    positive       0.85      0.80      0.83      2151

    accuracy                           0.73      6548
   macro avg       0.74      0.73      0.73      6548
weighted avg       0.74      0.73      0.73      6548

--------------------------------------------------------------------------------


(CatBoostClassifier(depth=8, devices='0', iterations=800, learning_rate=0.05, random_seed=42, task_type='GPU', verbose=0),
 array([[0],
        [2],
        [0],
        ...,
        [1],
        [1],
        [0]], shape=(6548, 1)))

## Эксперимент 5 — LogReg с перебором гиперпараметров



In [ ]:
param_grid = {
    'C':            [0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    'class_weight': [None, 'balanced']
}

gs = GridSearchCV(
    LogisticRegression(max_iter=2000, random_state=SEED, solver='lbfgs'),
    param_grid,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1,
    verbose=1
)

print('Запускаем GridSearch для LogReg...')
gs.fit(X_train_tfidf, y_train)

print(f'Лучшие параметры: {gs.best_params_}')
print(f'Лучший CV Macro F1: {gs.best_score_:.4f}')

evaluate(
    'LogReg (GridSearch)',
    gs.best_estimator_,
    X_train_tfidf, y_train,
    X_val_tfidf, y_val,
    'Подбор C и class_weight',
    str(gs.best_params_)
)

Запускаем GridSearch для LogReg...
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Лучшие параметры: {'C': 0.5, 'class_weight': 'balanced'}
Лучший CV Macro F1: 0.7409
[LogReg (GridSearch)] Macro F1 = 0.7488 | Weighted F1 = 0.7482 | Acc = 0.7462

Classification Report:
              precision    recall  f1-score   support

    negative       0.75      0.70      0.73      2224
     neutral       0.63      0.69      0.66      2173
    positive       0.88      0.85      0.86      2151

    accuracy                           0.75      6548
   macro avg       0.75      0.75      0.75      6548
weighted avg       0.75      0.75      0.75      6548

--------------------------------------------------------------------------------


(LogisticRegression(C=0.5, class_weight='balanced', max_iter=2000,
                    random_state=42),
 array([0, 2, 0, ..., 0, 1, 0], shape=(6548,)))

## Эксперимент 5b — LinearSVC (calibrated)


In [ ]:
svc_base       = LinearSVC(C=1.0, max_iter=3000, random_state=SEED, class_weight='balanced')
svc_calibrated = CalibratedClassifierCV(svc_base, cv=3, method='sigmoid')

evaluate(
    'LinearSVC (calibrated)',
    svc_calibrated,
    X_train_tfidf, y_train,
    X_val_tfidf, y_val,
    'Linear SVM часто превосходит LogReg на TF-IDF',
    'C=1.0, balanced, calibrated'
)

[LinearSVC (calibrated)] Macro F1 = 0.7361 | Weighted F1 = 0.7355 | Acc = 0.7352

Classification Report:
              precision    recall  f1-score   support

    negative       0.72      0.71      0.72      2224
     neutral       0.62      0.63      0.63      2173
    positive       0.87      0.86      0.86      2151

    accuracy                           0.74      6548
   macro avg       0.74      0.74      0.74      6548
weighted avg       0.74      0.74      0.74      6548

--------------------------------------------------------------------------------


(CalibratedClassifierCV(cv=3,
                        estimator=LinearSVC(class_weight='balanced',
                                            max_iter=3000, random_state=42)),
 array([0, 2, 0, ..., 0, 1, 0], shape=(6548,)))

## Эксперимент 5c — Char n-grams + Combined features



In [ ]:
tfidf_char = TfidfVectorizer(
    max_features=25000,
    ngram_range=(3, 6),
    analyzer='char_wb',
    min_df=3,
    sublinear_tf=True
)

X_train_char = tfidf_char.fit_transform(X_train_text)
X_val_char   = tfidf_char.transform(X_val_text)

X_train_combined = hstack([X_train_tfidf, X_train_char])
X_val_combined   = hstack([X_val_tfidf, X_val_char])

print(f'Char TF-IDF shape:  {X_train_char.shape}')
print(f'Combined shape:     {X_train_combined.shape}')

Char TF-IDF shape:  (37104, 25000)
Combined shape:     (37104, 75000)


In [1]:
lr_combined = LogisticRegression(
    C=2.0,
    solver='saga',   # вместо дефолтного 'lbfgs' — быстрее на больших разреженных матрицах
    max_iter=1000,
    n_jobs=-1,
    random_state=SEED
)
evaluate(
    'LogReg + Word+Char ngrams',
    lr_combined,
    X_train_combined, y_train,
    X_val_combined, y_val,
    'Комбинация word и char n-grams',
    'C=2.0'
)

# CPU-режим — n_jobs=-1 безопасен (нет GPU-конфликта)
lgbm_combined = lgb.LGBMClassifier(
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=127,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)
evaluate(
    'LightGBM + Word+Char',
    lgbm_combined,
    X_train_combined, y_train,
    X_val_combined, y_val,
    'Бустинг на комбинированных фичах',
    'n=800, leaves=127'
)

NameError: name 'LogisticRegression' is not defined

## Эксперимент 5d — LightGBM Tuned (усиленный)



In [2]:
lgbm_tuned = lgb.LGBMClassifier(
    n_estimators=500,          # было 1200 — слишком долго на CPU
    learning_rate=0.05,        # было 0.03 — при меньшем lr нужно больше деревьев
    num_leaves=127,            # было 255 — огромная нагрузка на 50k фичах
    min_data_in_leaf=20,
    feature_fraction=0.8,
    subsample=0.85,
    subsample_freq=5,          # активирует bagging
    colsample_bytree=0.85,
    random_state=SEED,
    n_jobs=1,
    verbose=-1,
    device='gpu',            # раскомментируй при наличии GPU
    gpu_platform_id=0,
    gpu_device_id=0,
)
evaluate(
    'LightGBM Tuned',
    lgbm_tuned,
    X_train_tfidf, y_train,
    X_val_tfidf, y_val,
    'Более мощный LightGBM',
    'n=500, lr=0.05, leaves=127, ff=0.8, sub=0.85'
)


NameError: name 'lgb' is not defined

## Эксперимент 6 — Уменьшение размерности (SVD / LSA)


In [13]:
for n_components in [100, 300]:
    svd = TruncatedSVD(n_components=n_components, random_state=SEED)
    X_tr_svd = svd.fit_transform(X_train_tfidf)
    X_v_svd  = svd.transform(X_val_tfidf)

    lr_svd = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    lr_svd.fit(X_tr_svd, y_train)
    preds = lr_svd.predict(X_v_svd)

    f1_macro    = f1_score(y_val, preds, average='macro')
    f1_weighted = f1_score(y_val, preds, average='weighted')
    acc         = accuracy_score(y_val, preds)

    name = f'LSA (SVD {n_components}) + LogReg'
    results.append({
        'Модель':          name,
        'Гипотеза':        'SVD выделит темы, улучшит качество',
        'Параметры':       f'n_components={n_components}',
        'Val Macro F1':    round(f1_macro, 4),
        'Val Weighted F1': round(f1_weighted, 4),   # исправлено: раньше отсутствовало
        'Val Accuracy':    round(acc, 4),            # исправлено: раньше отсутствовало
    })
    model_registry[name] = lr_svd

    print(f'[SVD {n_components}] Macro F1 = {f1_macro:.4f} | Weighted F1 = {f1_weighted:.4f} | Acc = {acc:.4f}')


[SVD 100] Macro F1 = 0.7103 | Weighted F1 = 0.7098 | Acc = 0.7075
[SVD 300] Macro F1 = 0.7339 | Weighted F1 = 0.7334 | Acc = 0.7314


## Эксперимент 7 — Ансамбль (Voting)


In [ ]:
lr_ens = LogisticRegression(
    C=gs.best_params_['C'],
    class_weight=gs.best_params_['class_weight'],
    max_iter=2000,             
    random_state=SEED
)
lgbm_ens = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)
cb_ens = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    task_type='GPU',          
    devices='0',
    random_seed=SEED,
    verbose=0
)

voting = VotingClassifier(
    estimators=[('lr', lr_ens), ('lgbm', lgbm_ens), ('cb', cb_ens)],
    voting='soft',
)
evaluate(
    'Voting (LogReg + LightGBM + CatBoost)', voting,
    X_train_tfidf, y_train, X_val_tfidf, y_val,
    'Объединение моделей даёт прирост', 'soft voting, sequential fit'
)
